# 03 · Deformation vs Moisture: Fresnel Phase & Phase Triplets

Reproduces:
- **Fig. 5** — Phase of the one-way Fresnel transmission coefficient (TE and TM)
  vs soil moisture (De Zan 2014).
- **Fig. 8** — Predicted phase triplets (closure phases) for all combinations
  of 7 acquisitions (De Zan 2014).

Key insight: a pure deformation term cancels in triplets; soil moisture does not.

---
**References**
- De Zan, F., Parizzi, A., Prats-Iraola, P., & López-Dekker, P. (2014).
  *A SAR interferometric model for soil moisture.*
  IEEE Transactions on Geoscience and Remote Sensing, 52(1), 418–425.
  https://doi.org/10.1109/TGRS.2013.2241069

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from navasar.coherence import fresnel_phase, phase_triplet

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
KW = dict(freq_ghz=1.4, theta_inc_deg=45.0, sand=0.51, clay=0.13)

## Fig. 5 — Fresnel transmission phase (De Zan 2014)

In [ ]:
mv = np.linspace(0.0, 0.5, 200)
phase_te, phase_tm = fresnel_phase(mv, **KW)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(mv, phase_te, 'b-',  label='TE (HH)')
ax.plot(mv, phase_tm, 'r--', label='TM (VV)')
ax.set_xlabel('Volumetric moisture $m_v$')
ax.set_ylabel('Transmission phase [deg]')
ax.set_title('One-way Fresnel transmission phase\n(De Zan 2014, Fig. 5)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig05_fresnel_phase.png', dpi=150)
plt.show()
print('Max TE phase:', np.max(np.abs(phase_te)), 'deg  → negligible vs model phases')

## Fig. 8 — Phase triplets for AGRISAR-like moisture series (De Zan 2014)

We use the 7 moisture values inferred from the AGRISAR 2006 campaign
(May 11 – July 5, field 222, L-band HH+VV) as reported in the paper.

In [ ]:
# Approximate in-situ moisture values from AGRISAR 2006 (Fig. 12 of De Zan 2014)
# Dates: May-11, May-24, Jun-06, Jun-13, Jun-21, Jul-05
dates_doy = np.array([131, 144, 157, 164, 172, 186])
mv_agrisar = np.array([0.28, 0.22, 0.18, 0.15, 0.12, 0.10])

n = len(mv_agrisar)
triplets_pred = []
triplet_labels = []

for i, j, k in combinations(range(n), 3):
    tp = phase_triplet(mv_agrisar[i], mv_agrisar[j], mv_agrisar[k], **KW)
    triplets_pred.append(tp)
    triplet_labels.append(f'{i+1}-{j+1}-{k+1}')

triplets_pred = np.array(triplets_pred)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(triplets_pred)), triplets_pred, color='steelblue', alpha=0.8)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(range(len(triplet_labels)))
ax.set_xticklabels(triplet_labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Phase triplet [deg]')
ax.set_title('Predicted phase triplets — AGRISAR 2006 moisture series\n(De Zan 2014, Fig. 8)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/fig08_phase_triplets.png', dpi=150)
plt.show()

## Why triplets are non-zero: non-linearity of the model

The triplet $\phi_{1,2,3} = \phi_{12} + \phi_{23} - \phi_{13}$ is non-zero
because the interferometric phase is a **non-linear** function of $m_v$.

To show this we need **three distinct moisture values** — if $m_{v1} = m_{v3}$
the triplet is always zero by symmetry regardless of $m_{v2}$.
Here we fix $m_{v1}$ (master) and $m_{v3}$ (second slave) at different
baselines and vary $m_{v2}$ (middle acquisition) to show the non-zero response.

In [ ]:
mv2_range = np.linspace(0.01, 0.50, 200)

# Three asymmetric cases: mv1 ≠ mv3, so triplet is genuinely non-zero
cases = [
    (0.10, 0.30, 'mv1=0.10, mv3=0.30'),
    (0.15, 0.25, 'mv1=0.15, mv3=0.25'),
    (0.05, 0.40, 'mv1=0.05, mv3=0.40'),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: triplet vs mv2 for fixed asymmetric mv1/mv3
for mv1, mv3, label in cases:
    tp = [phase_triplet(mv1, mv2, mv3, **KW) for mv2 in mv2_range]
    axes[0].plot(mv2_range, tp, label=label)
axes[0].axhline(0, color='k', lw=0.8, ls='--')
axes[0].set_xlabel('Middle acquisition moisture $m_{v2}$')
axes[0].set_ylabel('Phase triplet [deg]')
axes[0].set_title('Triplet vs $m_{v2}$ for asymmetric mv1/mv3\n'                  '(non-zero because model is non-linear in $m_v$)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Right: 2-D map — triplet as function of mv1 and mv3, mv2 fixed at 0.30
mv_grid = np.linspace(0.01, 0.50, 80)
MV1, MV3 = np.meshgrid(mv_grid, mv_grid)
tp_map = phase_triplet(MV1, 0.30, MV3, **KW)
im = axes[1].contourf(mv_grid, mv_grid, tp_map, levels=20, cmap='RdBu_r')
axes[1].contour(mv_grid, mv_grid, tp_map, levels=[0], colors='k', linewidths=1)
plt.colorbar(im, ax=axes[1], label='Triplet [deg]')
axes[1].set_xlabel('$m_{v1}$ (master)')
axes[1].set_ylabel('$m_{v3}$ (second slave)')
axes[1].set_title('Triplet map: $m_{v2}$ = 0.30 fixed\n'                  'Zero only on diagonal $m_{v1}=m_{v3}$ (white line)')
axes[1].plot([0.01,0.50],[0.01,0.50],'k--',lw=0.8)
axes[1].grid(True, alpha=0.2)

plt.suptitle('Non-linearity of the De Zan model — why triplets are non-zero', y=1.02)
plt.tight_layout()
plt.savefig('../examples/fig08b_triplet_nonlinearity.png', dpi=150)
plt.show()

# Sanity check: symmetric case must be zero
tp_sym = phase_triplet(0.20, 0.35, 0.20, **KW)
tp_asym = phase_triplet(0.10, 0.35, 0.20, **KW)
print(f'Symmetric  (mv1=mv3=0.20, mv2=0.35): triplet = {tp_sym:.4f} deg  ← must be 0')
print(f'Asymmetric (mv1=0.10, mv2=0.35, mv3=0.20): triplet = {tp_asym:.2f} deg  ← non-zero')